# Warren/Cena IMPROVE Prep Analysis

This notebook turns the meeting notes into a compact technical package for the Warren White and Cena discussions.

The main framing is deliberately first-order:

1. Start with the SPARTAN result: Addis/ETAD behaves differently in HIPS `fAbs` vs FTIR EC than Beijing, Delhi, and JPL.
2. Use IMPROVE as the long-record reference for whether Addis-like HIPS absorption and EC filter loading are common measurement regimes.
3. Compare each axis separately before interpreting slope/intercept: `fAbs`, EC concentration, EC mass on filter, and EC surface loading.
4. Treat RT/blank-line plots as diagnostic caveats because FED does not expose the raw blank rows and lot-specific coefficients needed for a full SPARTAN-style recalculation.

Date note: the IMPROVE stable-period filter is implemented as `year >= 2003`. That follows the literature/meeting language of consistent calibration `2003-present` / stable calibration `since 2003`; it is not a claim that the exact transition occurred on January 1, 2003.

## Discussion-Derived To-Do List

- Show the core cross-site HIPS/FTIR EC relationships clearly, with Addis separated enough that non-project readers can understand the anomaly.
- Avoid using moving high-`fAbs` thresholds as the main analysis because that selects on the response variable and inflates intercept behavior.
- Ask whether IMPROVE has filters comparable to ETAD/Addis by `fAbs`, EC mass on the filter, and EC surface loading.
- Focus on mass/surface loading first; keep iron, soil, and source composition as second-order checks.
- Use FED RT fields only as proxies. For Warren, the main question is whether raw HIPS blanks/coefficient records are needed to interpret Addis.
- For Cena, the main operational question is whether the Addis aethalometer flow-ratio fix was actually implemented and whether post-fix data exist.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path('/Users/ahmadjalil/github/aethmodular')
SPARTAN_DIR = PROJECT_DIR / 'research' / 'ftir_hips_chem'
IMPROVE_CLEAN_PATH = SPARTAN_DIR / 'output' / 'improve_high_fabs_comparison' / 'improve_valid_cleaned.csv'
OUT_DIR = PROJECT_DIR / 'research' / 'improve_hips_offset' / 'output' / 'warren_cena_improve_prep_analysis'
OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SPARTAN_DIR / 'scripts'))
from config import MAC_VALUE
from outliers import apply_exclusion_flags, apply_threshold_flags
from plotting import apply_default_style

apply_default_style()
warnings.filterwarnings('ignore', category=FutureWarning)

STABLE_YEAR = 2003
PRIMARY_IMPROVE_AREA_CM2 = 3.5
SPARTAN_FILTER_PATH = SPARTAN_DIR / 'Filter Data' / 'unified_filter_dataset.pkl'
SITE_CODE_TO_NAME = {'CHTS': 'Beijing', 'INDH': 'Delhi', 'USPA': 'JPL', 'ETAD': 'Addis_Ababa'}
SITE_ORDER = ['Beijing', 'Delhi', 'JPL', 'Addis_Ababa']
SITE_COLORS = {'Beijing': '#1f77b4', 'Delhi': '#d62728', 'JPL': '#2ca02c', 'Addis_Ababa': '#9467bd'}

print(f'Outputs: {OUT_DIR}')

## Load and Clean SPARTAN HIPS/FTIR Reference

This cell rebuilds the PM2.5 HIPS/FTIR pairing from `unified_filter_dataset.pkl`, applies the project exclusion registry, and keeps the HIPS raw T/R fields needed for the local blank/lot caveat plot.

In [ ]:
def regression_stats(df, x_col, y_col):
    d = df[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()
    d = d[(d[x_col] > 0) & (d[y_col] > 0)]
    if len(d) < 3:
        return {'n': len(d), 'slope': np.nan, 'intercept': np.nan, 'r2': np.nan, 'origin_slope': np.nan}
    x = d[x_col].to_numpy(float)
    y = d[y_col].to_numpy(float)
    slope, intercept = np.polyfit(x, y, 1)
    pred = slope * x + intercept
    ss_res = np.sum((y - pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return {
        'n': len(d),
        'slope': slope,
        'intercept': intercept,
        'r2': 1 - ss_res / ss_tot if ss_tot > 0 else np.nan,
        'origin_slope': np.sum(x * y) / np.sum(x ** 2),
    }


def load_spartan_clean():
    df_long = pd.read_pickle(SPARTAN_FILTER_PATH)
    df_long['SampleDate'] = pd.to_datetime(df_long['SampleDate'], errors='coerce')
    df_long['SiteName'] = df_long['Site'].map(SITE_CODE_TO_NAME)

    params = [
        'EC_ftir', 'HIPS_Fabs', 'HIPS_tau', 'HIPS_T1', 'HIPS_R1', 'HIPS_t', 'HIPS_r',
        'ChemSpec_OC_PM2.5', 'ChemSpec_Filter_PM2.5_mass', 'ChemSpec_Iron_PM2.5',
    ]
    wide = (
        df_long[df_long['Parameter'].isin(params)]
        .pivot_table(
            index=['Site', 'SiteName', 'FilterId', 'SampleDate', 'FilterType', 'LotId', 'DepositArea_cm2', 'Volume_m3'],
            columns='Parameter',
            values='Concentration',
            aggfunc='first',
        )
        .reset_index()
    )
    wide.columns.name = None

    ec_mass = (
        df_long[df_long['Parameter'].eq('EC_ftir')][['FilterId', 'MassLoading_ug']]
        .drop_duplicates('FilterId')
        .rename(columns={'MassLoading_ug': 'EC_loading_ug'})
    )
    wide = wide.merge(ec_mass, on='FilterId', how='left')
    wide['HIPS_BC_MAC10'] = wide['HIPS_Fabs'] / MAC_VALUE
    wide['EC_loading_ug_cm2'] = wide['EC_loading_ug'] / wide['DepositArea_cm2']
    wide['t_plus_r'] = wide.get('HIPS_t', np.nan) + wide.get('HIPS_r', np.nan)

    clean_frames = []
    for site_name in SITE_ORDER:
        site_code = [code for code, label in SITE_CODE_TO_NAME.items() if label == site_name][0]
        site_df = wide[(wide['Site'].eq(site_code)) & (wide['FilterType'].eq('PM2.5'))].copy()
        site_df['aeth_bc'] = pd.NA
        site_df['filter_ec'] = site_df['EC_ftir'] * 1000.0
        site_df['date'] = site_df['SampleDate']
        site_df['filter_id'] = site_df['FilterId']
        site_df = apply_exclusion_flags(site_df, site_name)
        site_df = apply_threshold_flags(site_df, site_name)
        clean = site_df[
            (~site_df['is_excluded'])
            & (~site_df['is_outlier'])
            & site_df['EC_ftir'].notna()
            & site_df['HIPS_Fabs'].notna()
            & site_df['EC_loading_ug'].notna()
            & site_df['EC_loading_ug_cm2'].notna()
            & (site_df['EC_ftir'] > 0)
            & (site_df['HIPS_Fabs'] > 0)
        ].copy()
        clean_frames.append(clean)

    clean_pm25 = pd.concat(clean_frames, ignore_index=True)
    return wide, clean_pm25

spartan_all, spartan_clean = load_spartan_clean()
etad = spartan_clean[spartan_clean['SiteName'].eq('Addis_Ababa')].copy()

spartan_fit_rows = []
for site, g in spartan_clean.groupby('SiteName'):
    row = {'SiteName': site, **regression_stats(g, 'EC_ftir', 'HIPS_Fabs')}
    row['EC_loading_ug_cm2_median'] = g['EC_loading_ug_cm2'].median()
    row['HIPS_Fabs_median'] = g['HIPS_Fabs'].median()
    spartan_fit_rows.append(row)
spartan_fits = pd.DataFrame(spartan_fit_rows).set_index('SiteName').reindex(SITE_ORDER).reset_index()
spartan_fits.to_csv(OUT_DIR / 'spartan_hips_fabs_vs_ec_fits.csv', index=False)

display(spartan_fits.round(3))

## Core SPARTAN Cross-Site Result

This is the figure to start the technical story: Addis has a high `fAbs` regime and a different HIPS/EC relationship from the lower-loading sites. The dotted lines are MAC reference slopes in `Mm^-1 per ug/m3`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True)
axes = axes.ravel()
xmax = max(13, spartan_clean['EC_ftir'].quantile(0.99) * 1.05)
ymax = max(95, spartan_clean['HIPS_Fabs'].quantile(0.99) * 1.05)
xline = np.linspace(0, xmax, 100)

for ax, site in zip(axes, SITE_ORDER):
    g = spartan_clean[spartan_clean['SiteName'].eq(site)]
    fit = regression_stats(g, 'EC_ftir', 'HIPS_Fabs')
    ax.scatter(g['EC_ftir'], g['HIPS_Fabs'], s=36, alpha=0.75, color=SITE_COLORS[site], edgecolor='white', linewidth=0.35)
    if np.isfinite(fit['slope']):
        ax.plot(xline, fit['slope'] * xline + fit['intercept'], color='black', lw=1.5, label='OLS')
    ax.plot(xline, 10 * xline, color='0.35', lw=1.1, ls=':', label='MAC=10')
    ax.plot(xline, 20 * xline, color='0.55', lw=1.0, ls=':', label='MAC=20')
    ax.set_title(f'{site}\nn={fit["n"]}, slope={fit["slope"]:.2f}, intercept={fit["intercept"]:.1f}, R2={fit["r2"]:.2f}')
    ax.set_xlabel('FTIR EC (ug/m3)')
    ax.set_ylabel('HIPS fAbs (Mm$^{-1}$)')
    ax.set_xlim(0, xmax)
    ax.set_ylim(0, ymax)
    ax.legend(frameon=False, fontsize=8)

fig.suptitle('SPARTAN HIPS fAbs vs FTIR EC: Addis is the high-loading anomaly', y=1.01)
fig.tight_layout()
fig.savefig(OUT_DIR / 'spartan_four_site_hips_fabs_vs_ec.png', dpi=230, bbox_inches='tight')
plt.show()

## ETAD/Addis First-Order Reference Ranges

These ranges define the physical comparison target for IMPROVE. The p05-p95 bounds are the primary comparison; min-max bounds are an audit.

In [ ]:
range_specs = {
    'HIPS_Fabs_Mm-1': 'HIPS_Fabs',
    'EC_ug_m3': 'EC_ftir',
    'EC_loading_ug_filter': 'EC_loading_ug',
    'EC_loading_ug_cm2': 'EC_loading_ug_cm2',
}
rows = []
for label, col in range_specs.items():
    s = etad[col].dropna()
    rows.append({
        'metric': label,
        'n': len(s),
        'min': s.min(),
        'p05': s.quantile(.05),
        'p25': s.quantile(.25),
        'median': s.median(),
        'p75': s.quantile(.75),
        'p95': s.quantile(.95),
        'max': s.max(),
    })
etad_bounds = pd.DataFrame(rows)
etad_bounds.to_csv(OUT_DIR / 'etad_first_order_reference_ranges.csv', index=False)
display(etad_bounds.round(3))

## Load IMPROVE FED Clean Pull

The cleaned FED table already has positive EC + `fAbs`, joined site metadata, flow/duration-derived sampled volume, EC mass loading, surface-loading sensitivities, and available RT fields.

In [ ]:
usecols = [
    'Dataset', 'SiteCode', 'POC', 'Date', 'AuxID', 'ECf_Val', 'OCf_Val', 'fAbs_Val',
    'FlowRate_Val', 'SampDur_Val', 'FEf_Val', 'MF_Val', 'SOILf_Val', 'volume_m3',
    'EC_loading_ug', 'EC_loading_ug_cm2_primary', 'fAbs_per_EC', 'OC_EC', 'FE_EC', 'SOIL_EC',
    'year', 'month', 'SiteName', 'Country', 'State', 'County', 'Latitude', 'Longitude',
    'RefF_635_Val', 'TransF_635_Val', 'RefI_635_Val', 'TransI_635_Val', 'RefM_635_Val', 'TransM_635_Val', 'rt_available',
]
improve = pd.read_csv(IMPROVE_CLEAN_PATH, usecols=lambda c: c in set(usecols), parse_dates=['Date'])
improve['year'] = improve['Date'].dt.year
improve['month'] = improve['Date'].dt.month
improve['post_2003'] = improve['year'] >= STABLE_YEAR
improve['valid_loading'] = improve['EC_loading_ug'].notna() & improve['EC_loading_ug_cm2_primary'].notna()
improve['rt_available'] = improve['rt_available'].fillna(False).astype(bool)
post = improve[improve['post_2003']].copy()

print(f'All cleaned IMPROVE rows: {len(improve):,}')
print(f'Post-2003 rows: {len(post):,}')
print(f'Post-2003 valid loading rows: {post["valid_loading"].sum():,}')
print(f'Post-2003 RT rows: {post["rt_available"].sum():,}')
display(post[['ECf_Val', 'fAbs_Val', 'volume_m3', 'EC_loading_ug', 'EC_loading_ug_cm2_primary']].describe(percentiles=[.05, .5, .95, .99, .999]).T.round(3))

## IMPROVE Baseline Before Screening

This answers the advisor request to look at all post-2003 data before removing low values or isolating the high tail.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
valid = post[post['valid_loading']].copy()

hb = axes[0].hexbin(valid['ECf_Val'], valid['fAbs_Val'], gridsize=95, bins='log', cmap='Greys', mincnt=1)
axes[0].plot(np.linspace(0, 12, 100), 10 * np.linspace(0, 12, 100), color='#2563EB', ls=':', lw=1.2, label='MAC=10')
axes[0].plot(np.linspace(0, 12, 100), 20 * np.linspace(0, 12, 100), color='#7C3AED', ls=':', lw=1.1, label='MAC=20')
axes[0].axhspan(etad_bounds.loc[etad_bounds.metric.eq('HIPS_Fabs_Mm-1'), 'p05'].iloc[0], etad_bounds.loc[etad_bounds.metric.eq('HIPS_Fabs_Mm-1'), 'p95'].iloc[0], color='#F59E0B', alpha=0.14, label='ETAD fAbs p05-p95')
axes[0].set_xlim(0, max(12, valid['ECf_Val'].quantile(.995)))
axes[0].set_ylim(0, max(100, valid['fAbs_Val'].quantile(.999)))
axes[0].set_xlabel('IMPROVE ECf (ug/m3)')
axes[0].set_ylabel('IMPROVE fAbs (Mm$^{-1}$)')
axes[0].set_title('Full post-2003 IMPROVE baseline')
axes[0].legend(frameon=False, fontsize=8)
fig.colorbar(hb, ax=axes[0], label='log10(count)')

axes[1].hist(valid['fAbs_Val'], bins=np.linspace(0, max(100, valid['fAbs_Val'].quantile(.999)), 100), color='#6B7280', alpha=0.85)
axes[1].axvspan(etad_bounds.loc[etad_bounds.metric.eq('HIPS_Fabs_Mm-1'), 'p05'].iloc[0], etad_bounds.loc[etad_bounds.metric.eq('HIPS_Fabs_Mm-1'), 'p95'].iloc[0], color='#F59E0B', alpha=0.18, label='ETAD fAbs p05-p95')
axes[1].set_yscale('log')
axes[1].set_xlabel('fAbs (Mm$^{-1}$)')
axes[1].set_ylabel('Rows (log)')
axes[1].set_title('High fAbs is a rare IMPROVE tail')
axes[1].legend(frameon=False, fontsize=8)

fig.tight_layout()
fig.savefig(OUT_DIR / 'improve_post2003_baseline_before_screening.png', dpi=230, bbox_inches='tight')
plt.show()

## Axis-by-Axis Comparability to ETAD/Addis

The key meeting pivot was to stop asking only about slope/intercept and first ask whether each measurement axis overlaps at all.

In [ ]:
def b(metric, field):
    return etad_bounds.set_index('metric').loc[metric, field]

post['etad_fabs_p05p95'] = post['fAbs_Val'].between(b('HIPS_Fabs_Mm-1', 'p05'), b('HIPS_Fabs_Mm-1', 'p95'), inclusive='both')
post['etad_fabs_minmax'] = post['fAbs_Val'].between(b('HIPS_Fabs_Mm-1', 'min'), b('HIPS_Fabs_Mm-1', 'max'), inclusive='both')
post['etad_ec_conc_p05p95'] = post['ECf_Val'].between(b('EC_ug_m3', 'p05'), b('EC_ug_m3', 'p95'), inclusive='both')
post['etad_ec_mass_p05p95'] = post['EC_loading_ug'].between(b('EC_loading_ug_filter', 'p05'), b('EC_loading_ug_filter', 'p95'), inclusive='both')
post['etad_ec_surface_p05p95'] = post['EC_loading_ug_cm2_primary'].between(b('EC_loading_ug_cm2', 'p05'), b('EC_loading_ug_cm2', 'p95'), inclusive='both')
post['all_first_order_p05p95'] = post['valid_loading'] & post['etad_fabs_p05p95'] & post['etad_ec_mass_p05p95'] & post['etad_ec_surface_p05p95']
post['all_first_order_plus_ec_conc_p05p95'] = post['all_first_order_p05p95'] & post['etad_ec_conc_p05p95']

count_rows = []
base = post[post['valid_loading']]
for label, mask in [
    ('valid loading, post-2003', base.index == base.index),
    ('ETAD fAbs p05-p95 only', base['etad_fabs_p05p95']),
    ('ETAD EC concentration p05-p95 only', base['etad_ec_conc_p05p95']),
    ('ETAD EC mass p05-p95 only', base['etad_ec_mass_p05p95']),
    ('ETAD EC surface p05-p95 only', base['etad_ec_surface_p05p95']),
    ('fAbs + EC mass + EC surface', base['all_first_order_p05p95']),
    ('fAbs + EC conc + EC mass + EC surface', base['all_first_order_plus_ec_conc_p05p95']),
]:
    d = base[mask]
    count_rows.append({'screen': label, 'n': len(d), 'sites': d['SiteCode'].nunique(), 'first_year': d['year'].min(), 'last_year': d['year'].max()})
counts = pd.DataFrame(count_rows)
counts['pct_valid_loading'] = counts['n'] / len(base) * 100
counts.to_csv(OUT_DIR / 'improve_etad_axis_overlap_counts.csv', index=False)
display(counts.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
plot_counts = counts.iloc[1:].copy()
ax.barh(plot_counts['screen'][::-1], plot_counts['n'][::-1], color='#2563EB')
ax.set_xscale('log')
ax.set_xlabel('Rows retained (log scale)')
ax.set_title('Which ETAD/Addis axis is rare in IMPROVE?')
for i, (_, row) in enumerate(plot_counts.iloc[::-1].iterrows()):
    ax.text(max(row['n'], 1) * 1.08, i, f"{int(row['n']):,}", va='center', fontsize=9)
fig.tight_layout()
fig.savefig(OUT_DIR / 'improve_etad_axis_overlap_counts.png', dpi=220, bbox_inches='tight')
plt.show()

## Loading-Space View

This is the first-order plot for Warren: if the filters are not in the same EC mass/surface-loading regime, slope and intercept comparisons are secondary.

In [ ]:
comparable = post[post['all_first_order_p05p95']].copy()
sample = base.sample(min(60000, len(base)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.3), sharey=True)
for ax, xcol, xlabel, metric in [
    (axes[0], 'EC_loading_ug', 'EC mass on filter (ug)', 'EC_loading_ug_filter'),
    (axes[1], 'EC_loading_ug_cm2_primary', 'EC surface loading (ug/cm$^2$)', 'EC_loading_ug_cm2'),
]:
    ax.scatter(sample[xcol], sample['fAbs_Val'], s=7, alpha=0.08, color='#4B5563', rasterized=True, label='post-2003 IMPROVE')
    ax.scatter(comparable[xcol], comparable['fAbs_Val'], s=50, color='#C2410C', edgecolor='black', linewidth=0.35, label='ETAD p05-p95 on fAbs + loading')
    ax.axhspan(b('HIPS_Fabs_Mm-1', 'p05'), b('HIPS_Fabs_Mm-1', 'p95'), color='#F59E0B', alpha=0.14)
    ax.axvspan(b(metric, 'p05'), b(metric, 'p95'), color='#2563EB', alpha=0.10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('fAbs (Mm$^{-1}$)')
    ax.set_ylim(0, max(100, base['fAbs_Val'].quantile(.999)))
    ax.legend(frameon=False, fontsize=8)
axes[0].set_xlim(0, max(90, base['EC_loading_ug'].quantile(.995)))
axes[1].set_xlim(0, max(28, base['EC_loading_ug_cm2_primary'].quantile(.995)))
fig.suptitle('IMPROVE filters in ETAD/Addis loading space', y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / 'improve_fabs_vs_ec_loading_space.png', dpi=230, bbox_inches='tight')
plt.show()

comparable.to_csv(OUT_DIR / 'improve_first_order_comparable_rows.csv', index=False)

## Bounded Group Fits

These fits are descriptive. The important distinction is whether the bounded group was selected on physical loading, on `fAbs`, or on all axes together.

In [ ]:
groups = {
    'full_post2003_valid_loading': base,
    'ETAD_fAbs_only': base[base['etad_fabs_p05p95']],
    'ETAD_EC_mass_only': base[base['etad_ec_mass_p05p95']],
    'ETAD_EC_surface_only': base[base['etad_ec_surface_p05p95']],
    'ETAD_all_first_order': comparable,
}
fit_rows = []
for name, g in groups.items():
    row = {'group': name, **regression_stats(g, 'ECf_Val', 'fAbs_Val')}
    row['sites'] = g['SiteCode'].nunique()
    row['median_fAbs'] = g['fAbs_Val'].median()
    row['median_EC'] = g['ECf_Val'].median()
    row['median_EC_loading_ug_cm2'] = g['EC_loading_ug_cm2_primary'].median()
    fit_rows.append(row)
improve_fits = pd.DataFrame(fit_rows)
improve_fits.to_csv(OUT_DIR / 'improve_bounded_group_fits.csv', index=False)
display(improve_fits.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2), sharex=True, sharey=True)
plot_pairs = [('full_post2003_valid_loading', groups['full_post2003_valid_loading'], axes[0], '#4B5563'), ('ETAD_all_first_order', groups['ETAD_all_first_order'], axes[1], '#C2410C')]
xmax = max(12, base['ECf_Val'].quantile(.995))
ymax = max(100, base['fAbs_Val'].quantile(.999))
xline = np.linspace(0, xmax, 100)
for name, g, ax, color in plot_pairs:
    draw = g.sample(min(50000, len(g)), random_state=42) if len(g) else g
    ax.scatter(draw['ECf_Val'], draw['fAbs_Val'], s=8 if len(draw) > 1000 else 45, alpha=0.10 if len(draw) > 1000 else 0.85, color=color, edgecolor='none', rasterized=True)
    fit = regression_stats(g, 'ECf_Val', 'fAbs_Val')
    if np.isfinite(fit['slope']):
        ax.plot(xline, fit['slope'] * xline + fit['intercept'], color='black', lw=1.4, label=f"OLS y={fit['slope']:.2f}x+{fit['intercept']:.1f}")
    ax.plot(xline, 10 * xline, color='#2563EB', ls=':', lw=1.1, label='MAC=10')
    ax.plot(xline, 20 * xline, color='#7C3AED', ls=':', lw=1.0, label='MAC=20')
    ax.set_title(f'{name}\nn={fit["n"]:,}, R2={fit["r2"]:.2f}')
    ax.set_xlabel('IMPROVE ECf (ug/m3)')
    ax.set_ylabel('IMPROVE fAbs (Mm$^{-1}$)')
    ax.set_xlim(0, xmax)
    ax.set_ylim(0, ymax)
    ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / 'improve_fabs_ec_full_vs_comparable.png', dpi=230, bbox_inches='tight')
plt.show()

## SPARTAN T/R and Lot Caveat

This is a Figure-2-style diagnostic for the SPARTAN data, not a reproduction of Warren et al. Figure 2. Warren's Figure 2 plots registered raw HIPS sphere and plate detector outputs (`R`, `T`) for routine IMPROVE field blanks and active samples from October 2019 through April 2020, with the blank OLS line defining the no-absorption relationship.

The SPARTAN table has `HIPS_R1` and `HIPS_T1` plus `FilterType` and `LotId`, so we can make an analogous plot for our filters and see lot-separated blank/sample structure. That is directly useful for asking Warren/Cena about lot-specific calibration. It should not be interpreted as an IMPROVE Figure 2 recreation.

### Figure 2 Replot Feasibility

Current data status:

- **Exact Warren et al. Figure 2 from IMPROVE:** not possible from the current local FED pull alone. The paper states that reported HIPS absorption coefficients are public, but the sampled air volumes and individual sphere/plate signals are available from the authors on request. The FED export we have includes `RefF_635`, `TransF_635`, `RefI_635`, `TransI_635`, `RefM_635`, and `TransM_635`, but not the full field-blank/sample raw record with the same status/lot structure used in Figure 2.
- **SPARTAN analog:** possible and included below using `HIPS_R1` and `HIPS_T1`, with field blanks marked by `FilterType == 'FB'` and grouped by `LotId`.
- **IMPROVE FED RT proxy plot:** possible and included later, but it should be labeled as a proxy/diagnostic because it uses exported RT parameters rather than the raw registered Figure-2 inputs.

In [ ]:
rt = spartan_all[spartan_all['HIPS_T1'].notna() & spartan_all['HIPS_R1'].notna() & spartan_all['SiteName'].isin(['JPL', 'Addis_Ababa'])].copy()
rt['is_blank'] = rt['FilterType'].eq('FB')

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.4), sharex=False, sharey=False)
for ax, site in zip(axes, ['JPL', 'Addis_Ababa']):
    g = rt[rt['SiteName'].eq(site)]
    for lot, lg in g.groupby('LotId', dropna=False):
        samples = lg[~lg['is_blank']]
        blanks = lg[lg['is_blank']]
        ax.scatter(samples['HIPS_R1'], samples['HIPS_T1'], s=32, alpha=0.55, label=f'{lot} samples')
        ax.scatter(blanks['HIPS_R1'], blanks['HIPS_T1'], s=75, marker='s', edgecolor='black', linewidth=0.8, alpha=0.95, label=f'{lot} blanks')
    ax.set_title(f'{site}: raw SPARTAN HIPS T1/R1 by filter lot')
    ax.set_xlabel('HIPS R1')
    ax.set_ylabel('HIPS T1')
    ax.legend(frameon=False, fontsize=7, ncol=1)
fig.tight_layout()
fig.savefig(OUT_DIR / 'spartan_rt_lot_caveat_jpl_addis.png', dpi=230, bbox_inches='tight')
plt.show()

lot_counts = (
    rt.groupby(['SiteName', 'LotId', 'FilterType'], dropna=False)
    .size()
    .rename('n')
    .reset_index()
    .sort_values(['SiteName', 'LotId', 'FilterType'])
)
lot_counts.to_csv(OUT_DIR / 'spartan_rt_lot_counts.csv', index=False)
display(lot_counts)

## FED RT Proxy Caveat

The available FED RT fields can show whether high-`fAbs`/loading-matched filters occupy unusual RT space. They cannot reproduce the raw Warren-style blank-line calibration without blank rows and coefficients.

In [ ]:
rt_imp = base[base['rt_available']].copy()
rt_imp['initial_t_over_r'] = (rt_imp['TransI_635_Val'] / rt_imp['RefI_635_Val']).replace([np.inf, -np.inf], np.nan)
rt_imp['min_t_over_r'] = (rt_imp['TransM_635_Val'] / rt_imp['RefM_635_Val']).replace([np.inf, -np.inf], np.nan)
rt_draw = rt_imp.dropna(subset=['initial_t_over_r', 'min_t_over_r', 'fAbs_Val'])
rt_draw = rt_draw.sample(min(60000, len(rt_draw)), random_state=42)
comp_rt = comparable[comparable['rt_available']].copy()
comp_rt['initial_t_over_r'] = (comp_rt['TransI_635_Val'] / comp_rt['RefI_635_Val']).replace([np.inf, -np.inf], np.nan)
comp_rt['min_t_over_r'] = (comp_rt['TransM_635_Val'] / comp_rt['RefM_635_Val']).replace([np.inf, -np.inf], np.nan)

fig, ax = plt.subplots(figsize=(7.2, 5.8))
sc = ax.scatter(rt_draw['initial_t_over_r'], rt_draw['min_t_over_r'], c=rt_draw['fAbs_Val'], s=8, alpha=0.16, cmap='viridis', rasterized=True)
ax.scatter(comp_rt['initial_t_over_r'], comp_rt['min_t_over_r'], s=58, facecolor='none', edgecolor='#F97316', linewidth=1.1, label='ETAD-comparable rows')
ax.set_xlabel('FED initial Trans/Ref proxy')
ax.set_ylabel('FED minimum Trans/Ref proxy')
ax.set_title('IMPROVE RT proxy space: diagnostic only')
ax.legend(frameon=False, fontsize=8)
fig.colorbar(sc, ax=ax, label='fAbs (Mm$^{-1}$)')
fig.tight_layout()
fig.savefig(OUT_DIR / 'improve_rt_proxy_caveat.png', dpi=230, bbox_inches='tight')
plt.show()

## Questions for Warren and Cena

### Warren White

1. Are the Addis/ETAD SPARTAN HIPS filters too heavily loaded for a defensible HIPS absorption interpretation?
2. If high EC surface loading is the issue, is there a published or internal correction appropriate for SPARTAN-style PTFE filters?
3. Does the IMPROVE `year >= 2003` comparison remain the right stable-calibration subset for this purpose?
4. What raw HIPS records would be needed to reproduce the blank-line / coefficient analysis for the candidate SPARTAN filters?
5. Is the apparent under-response/compression at Addis consistent with known loading, pixelation, or filter nonuniformity behavior, or does it point to a different mechanism?

### Cena / SPARTAN Operations

1. Was the Addis aethalometer flow-ratio correction physically implemented at the site, and on what exact date?
2. Is there post-fix aethalometer data with stable flow ratios that can be compared against earlier distributions even without matching filters?
3. Can SPARTAN provide raw HIPS blank/sample records, filter lot mapping, and the coefficients used for Addis and comparison sites?
4. Are Addis sampling duration or protocol changes feasible if filter loading is outside the validated HIPS range?

### Paper/Presentation Framing

- We have tested many second-order explanations; none resolves the Addis HIPS anomaly.
- The first-order question is now whether Addis sits outside the validated HIPS/filter-loading regime.
- IMPROVE provides a strong reference record, but ETAD/Addis-like high `fAbs` plus comparable EC loading appears rare enough that candidate rows need case-level review.

In [ ]:
slide_readout = pd.DataFrame([
    {'item': 'SPARTAN Addis clean PM2.5 filters', 'value': len(etad)},
    {'item': 'Addis fAbs p05-p95 (Mm-1)', 'value': f"{b('HIPS_Fabs_Mm-1', 'p05'):.2f}-{b('HIPS_Fabs_Mm-1', 'p95'):.2f}"},
    {'item': 'Addis EC surface loading p05-p95 (ug/cm2)', 'value': f"{b('EC_loading_ug_cm2', 'p05'):.2f}-{b('EC_loading_ug_cm2', 'p95'):.2f}"},
    {'item': 'IMPROVE post-2003 valid loading rows', 'value': int(len(base))},
    {'item': 'IMPROVE rows in Addis fAbs p05-p95 only', 'value': int(counts.loc[counts['screen'].eq('ETAD fAbs p05-p95 only'), 'n'].iloc[0])},
    {'item': 'IMPROVE rows in Addis EC mass p05-p95 only', 'value': int(counts.loc[counts['screen'].eq('ETAD EC mass p05-p95 only'), 'n'].iloc[0])},
    {'item': 'IMPROVE rows in Addis EC surface p05-p95 only', 'value': int(counts.loc[counts['screen'].eq('ETAD EC surface p05-p95 only'), 'n'].iloc[0])},
    {'item': 'IMPROVE rows in fAbs + EC mass + EC surface p05-p95', 'value': int(len(comparable))},
])
slide_readout.to_csv(OUT_DIR / 'slide_readout.csv', index=False)
display(slide_readout)

print('Key generated figures:')
for name in [
    'spartan_four_site_hips_fabs_vs_ec.png',
    'improve_post2003_baseline_before_screening.png',
    'improve_etad_axis_overlap_counts.png',
    'improve_fabs_vs_ec_loading_space.png',
    'improve_fabs_ec_full_vs_comparable.png',
    'spartan_rt_lot_caveat_jpl_addis.png',
    'improve_rt_proxy_caveat.png',
]:
    print('-', OUT_DIR / name)